# 1. Exploratory Data Analysis (EDA) and Preprocessing
This notebook covers the EDA and preprocessing pipeline for the AuraCart E-Commerce Analytics and MLOps System project.

## 1. Import Required Libraries
We import all necessary libraries for data analysis, visualization, and preprocessing.

In [ ]:
# Data manipulation and analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing and pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# For saving pipeline
import joblib

# Misc
import os

## 2. Project Requirements and Dataset Overview
- Perform EDA on the e-commerce dataset from Hugging Face.
- Build a robust, reusable preprocessing pipeline for downstream modeling.
- Save the pipeline for use in deployment.

**Dataset:** [millat/e-commerce-orders](https://huggingface.co/datasets/millat/e-commerce-orders)

**Key Features:**
- Numerical: price, quantity
- Categorical: category, payment_method, device_type, channel, customer_segment, delivery_status
- Date/time: order_date, shipping_date
- ID columns: order_id, customer_id, product_id (to be dropped)
- Address columns: shipping_address, billing_address (to be dropped or engineered)

We will:
- Analyze distributions, correlations, and class balance
- Engineer features from dates
- Encode categorical variables
- Scale numerical features
- Save the final pipeline

## 3. Download and Load the Dataset
We will load the dataset directly from Hugging Face using pandas.

In [ ]:
# Download and load the dataset from Hugging Face
df = pd.read_csv('https://huggingface.co/datasets/millat/e-commerce-orders/resolve/main/ecommerce_orders.csv')
df.head()

## 4. Initial Data Exploration
We will inspect the data types, missing values, and basic statistics.

In [ ]:
# Data info and missing values
df.info()
df.isnull().sum()

In [ ]:
# Basic statistics for numerical columns
df.describe()

In [ ]:
# Value counts for key categorical columns
categorical_cols = ['category', 'payment_method', 'device_type', 'channel', 'customer_segment', 'delivery_status']
for col in categorical_cols:
    print(f"\n{col} value counts:")
    print(df[col].value_counts())

## 5. Visualize Distributions and Correlations
We will plot histograms, boxplots, and a correlation heatmap for numerical features.

In [ ]:
# Histograms for numerical features
num_cols = ['price', 'quantity']
df[num_cols].hist(bins=30, figsize=(10, 4))
plt.suptitle('Histograms of Numerical Features')
plt.show()

# Boxplots for outlier detection
plt.figure(figsize=(10, 4))
for i, col in enumerate(num_cols):
    plt.subplot(1, 2, i+1)
    sns.boxplot(y=df[col])
    plt.title(f'Boxplot of {col}')
plt.tight_layout()
plt.show()

# Correlation heatmap
corr = df[num_cols].corr()
plt.figure(figsize=(5, 4))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

## 6. Feature Engineering and Data Cleaning
We will drop unnecessary columns, extract features from dates, and handle missing values.

In [ ]:
# Drop ID and address columns
drop_cols = ['order_id', 'customer_id', 'product_id', 'shipping_address', 'billing_address']
df_clean = df.drop(columns=drop_cols)

# Convert date columns to datetime
df_clean['order_date'] = pd.to_datetime(df_clean['order_date'])
df_clean['shipping_date'] = pd.to_datetime(df_clean['shipping_date'])

# Feature engineering: extract year, month, day, hour from order_date
for col in ['order_date', 'shipping_date']:
    df_clean[f'{col}_year'] = df_clean[col].dt.year
    df_clean[f'{col}_month'] = df_clean[col].dt.month
    df_clean[f'{col}_day'] = df_clean[col].dt.day
    df_clean[f'{col}_hour'] = df_clean[col].dt.hour

# Feature: days between order and shipping
df_clean['days_to_ship'] = (df_clean['shipping_date'] - df_clean['order_date']).dt.days

# Drop original date columns
df_clean = df_clean.drop(columns=['order_date', 'shipping_date'])

# Handle missing values (simple fill for demonstration)
df_clean = df_clean.fillna({'days_to_ship': df_clean['days_to_ship'].median()})
df_clean = df_clean.dropna()
df_clean.head()

## 7. Preprocessing Pipeline Construction
We will build a Scikit-learn pipeline to encode categorical variables and scale numerical features.

In [ ]:
# Identify columns for preprocessing
categorical_nominal = ['category', 'payment_method', 'device_type', 'channel']
categorical_ordinal = []  # No clear ordinal columns in this dataset
categorical_targets = ['customer_segment', 'delivery_status']
numerical = ['price', 'quantity', 'order_date_year', 'order_date_month', 'order_date_day', 'order_date_hour',
             'shipping_date_year', 'shipping_date_month', 'shipping_date_day', 'shipping_date_hour', 'days_to_ship']

# One-hot encode nominal categorical features
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical),
    ('cat_nom', OneHotEncoder(handle_unknown='ignore'), categorical_nominal)
], remainder='passthrough')

# Example: fit the pipeline on the cleaned data (excluding targets)
X = df_clean.drop(columns=categorical_targets)
preprocessor.fit(X)

# Transform the data
X_processed = preprocessor.transform(X)
X_processed.shape

## 8. Save the Preprocessing Pipeline
We will serialize the fitted pipeline for use in model training and deployment.

In [ ]:
# Save the fitted preprocessing pipeline
os.makedirs('../artifacts', exist_ok=True)
joblib.dump(preprocessor, '../artifacts/preprocessing_pipeline.joblib')
print('Preprocessing pipeline saved to ../artifacts/preprocessing_pipeline.joblib')

## 9. Summary of Key EDA Findings
- The dataset contains both numerical and categorical features, with some class imbalance in target variables.
- Feature engineering from dates and removal of ID/address columns improves data quality.
- The preprocessing pipeline is now ready for use in model training and deployment.